# 4. AHP Calculation
Analytical Hierarchy Process pairwise comparison and weight calculation for flood conditioning factors.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## 1. Parameters & Pairwise Matrix / 

Eight conditioning factors ordered by priority:
1. Rainfall (Rainfall) - primary trigger
2. Distance to River (Distance to River) - riverbank flood zone
3. Elevation (Elevation) - low-lying areas
4. Distance to Coast (Distance to Coast) - coastal flood, abrasion
5. Land Cover (Land Cover) - impermeable urban coastal areas
6. Slope (Slope) - slope, water flow
7. NDVI - deforestation upstream
8. Soil Type (Soil Type) - permeability

In [ ]:
# Pathlib Path for output directory
PROJECT_ROOT = Path.cwd().parent

params = [
    'Rainfall',
    'Distance to River',
    'Elevation',
    'Distance to Coast',
    'Land Cover',
    'Slope',
    'NDVI',
    'Soil Type'
]

# Pairwise comparison matrix (Saaty 1-9 scale)
# M[i][j] = importance of row-i vs column-j
M = np.array([
    # CH JS El JP TL Slp NDVI JT
    [1, 2, 3, 3, 3, 4, 4, 6], # Rainfall
    [1/2, 1, 2, 2, 2, 3, 3, 5], # Distance to River
    [1/3, 1/2, 1, 1, 2, 2, 3, 4], # Elevation
    [1/3, 1/2, 1, 1, 1, 2, 3, 4], # Distance to Coast
    [1/3, 1/2, 1/2, 1, 1, 2, 2, 3], # Land Cover
    [1/4, 1/3, 1/2, 1/2, 1/2, 1, 2, 2], # Slope
    [1/4, 1/3, 1/3, 1/2, 1/2, 1/2, 1, 2], # NDVI
    [1/6, 1/5, 1/4, 1/4, 1/3, 1/2, 1/2, 1], # Soil Type
], dtype=float)

## 2. Weight Calculation / 

In [ ]:
# Step 1: Column normalization
col_sums = M.sum(axis=0)
M_norm = M / col_sums

# Step 2: Weights = row averages of normalized matrix
weights = M_norm.mean(axis=1)

# Step 3: Consistency check
n = len(params)
weighted_sum = M @ weights
lambda_vec = weighted_sum / weights
lambda_max = lambda_vec.mean()
CI = (lambda_max - n) / (n - 1)
RI_table = {1:0, 2:0, 3:0.58, 4:0.90, 5:1.12, 6:1.24, 7:1.32, 8:1.41, 9:1.45, 10:1.49}
RI = RI_table[n]
CR = CI / RI

# Display results
results_df = pd.DataFrame({
    'Parameter': params,
    'Weight': [round(w, 4) for w in weights],
    'Percentage': [f'{w*100:.2f}%' for w in weights]
})
print('AHP Weight Results:')
print(results_df.to_string(index=False))
print(f'\nConsistency Check:')
print(f' Lambda max = {lambda_max:.4f}')
print(f' CI = {CI:.4f}')
print(f' RI (n={n}) = {RI:.2f}')
print(f' CR = {CR:.4f}')
print(f' CR < 0.10: {"Consistent" if CR < 0.10 else "INCONSISTENT - revise matrix"}')

## 3. Pairwise Comparison Matrix Display / 

In [ ]:
pairwise_df = pd.DataFrame(M, index=params, columns=[p[:6] for p in params])
plt.figure(figsize=(10, 8))
sns.heatmap(pairwise_df, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5)
plt.title('AHP Pairwise Comparison Matrix')
plt.tight_layout()
plt.show()

## 4. Weight Comparison Chart / 

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sorted_idx = np.argsort(weights)
ax.barh(range(len(sorted_idx)), [weights[i] for i in sorted_idx], color='steelblue')
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([params[i] for i in sorted_idx])
ax.set_xlabel('Weight')
ax.set_title('AHP Weights for Flood Conditioning Factors')
plt.tight_layout()
plt.show()

## 5. Save Outputs / 

In [ ]:
results_df.to_csv(PROJECT_ROOT / 'outputs' / 'ahp_weights.csv', index=False)

pairwise_out = pd.DataFrame(M, index=params, columns=params)
pairwise_out.to_csv(PROJECT_ROOT / 'outputs' / 'ahp_pairwise_matrix.csv')

print('Saved: outputs/ahp_weights.csv, outputs/ahp_pairwise_matrix.csv')